# Đo Params, GFLOPs, latency và FPS của 6 mô hình trên Tesla T4

Notebook này đo đồng nhất:

- YOLO26n, YOLO26s, YOLO26m, YOLO26l, YOLO26x bằng Ultralytics chính thức.
- Mamba-YOLO-B bằng mã nguồn Mamba-YOLO trong một tiến trình Python riêng.

Mamba-YOLO phải chạy riêng vì checkpoint sử dụng các lớp tùy chỉnh như
`ultralytics.nn.modules.mamba_yolo`, không có trong gói Ultralytics chính thức.

**Kaggle:** bật GPU T4 và bật Internet để notebook tự clone Mamba-YOLO. Khi không bật
Internet, hãy thêm repository Mamba-YOLO dưới dạng Kaggle Input.

## Bản vá PyTorch 2.6+

Notebook này đã xử lý lỗi `Weights only load failed` của checkpoint Mamba-YOLO.
PyTorch 2.6 trở lên chuyển mặc định của `torch.load` sang `weights_only=True`,
trong khi checkpoint này chứa cả đối tượng mô hình tùy chỉnh. Tiến trình Mamba
được cấu hình tải với `weights_only=False`.

Chỉ chạy bản vá này với checkpoint do chính bạn huấn luyện hoặc từ nguồn tin cậy.

In [9]:
# ===== [1] CÀI ĐẶT GÓI DÙNG CHUNG =====
# Không cài Mamba-YOLO đè lên Ultralytics chính thức.
!pip install -q -U ultralytics
!pip install -q seaborn thop timm einops dill

import os
import sys
import json
import glob
import re
import zipfile
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from ultralytics import YOLO

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA khả dụng:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("Notebook này cần bật GPU trên Kaggle.")

Python: 3.12.13
PyTorch: 2.10.0+cu128
CUDA khả dụng: True
GPU: Tesla T4


In [10]:
# ===== [2] TÌM VÀ GIẢI NÉN 6 FILE BEST.PT =====

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")
EXTRACT_ROOT = WORK_ROOT / "BEST_6_MODEL_extracted"
EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)

# Nếu BEST_6_MODEL được tải lên dưới dạng ZIP thì tự động giải nén.
for zip_path in INPUT_ROOT.rglob("*.zip"):
    try:
        with zipfile.ZipFile(zip_path) as zf:
            names = zf.namelist()
            if any("best_bestmamba.pt" in n.lower() for n in names):
                print("Giải nén:", zip_path)
                zf.extractall(EXTRACT_ROOT)
                break
    except zipfile.BadZipFile:
        pass

# Tìm tất cả checkpoint trong Input và thư mục vừa giải nén.
pt_paths = list(INPUT_ROOT.rglob("*.pt")) + list(EXTRACT_ROOT.rglob("*.pt"))

def identify_model(path):
    # Chuẩn hóa các tên file hiện có trong BEST_6_MODEL.zip.
    s = path.name.lower().replace("-", "_")

    if "mamba" in s:
        return "Mamba-YOLO-B"
    if "nano" in s or "yolo26n" in s:
        return "YOLO26n"
    if "small" in s or "yolo26s" in s:
        return "YOLO26s"
    if "mid" in s or "medium" in s or "yolo26m" in s:
        return "YOLO26m"
    if "xlarrge" in s or "xlarge" in s or "yolo26x" in s:
        return "YOLO26x"
    if "large" in s or "yolo26l" in s:
        return "YOLO26l"
    return None

MODEL_ORDER = [
    "YOLO26n", "YOLO26s", "YOLO26m",
    "YOLO26l", "YOLO26x", "Mamba-YOLO-B"
]

# Mỗi tên chỉ lấy một checkpoint.
model_map = {}
for p in pt_paths:
    name = identify_model(p)
    if name and name not in model_map:
        model_map[name] = p

missing = [name for name in MODEL_ORDER if name not in model_map]
if missing:
    raise FileNotFoundError(
        "Thiếu checkpoint: " + ", ".join(missing)
        + "\nHãy Add Data chứa BEST_6_MODEL.zip vào Kaggle Input."
    )

MODELS = [(name, str(model_map[name])) for name in MODEL_ORDER]

print("Đã tìm thấy đủ 6 mô hình:")
for name, path in MODELS:
    print(f"  {name:14s} -> {path}")

Đã tìm thấy đủ 6 mô hình:
  YOLO26n        -> /kaggle/input/datasets/nonlam123/best-6-model/BEST_6_MODEL/best_nano.pt
  YOLO26s        -> /kaggle/input/datasets/nonlam123/best-6-model/BEST_6_MODEL/best_small.pt
  YOLO26m        -> /kaggle/input/datasets/nonlam123/best-6-model/BEST_6_MODEL/best_mid.pt
  YOLO26l        -> /kaggle/input/datasets/nonlam123/best-6-model/BEST_6_MODEL/best_large.pt
  YOLO26x        -> /kaggle/input/datasets/nonlam123/best-6-model/BEST_6_MODEL/best_xlarrge.pt
  Mamba-YOLO-B   -> /kaggle/input/datasets/nonlam123/best-6-model/BEST_6_MODEL/best_bestmamba.pt


In [11]:
# ===== [3] TỰ TÌM DATASET VÀ TẠO DATASET.YAML =====

def find_dataset_root():
    candidates = []
    for test_images in INPUT_ROOT.rglob("test/images"):
        root = test_images.parent.parent
        train_ok = (root / "train/images").is_dir()
        val_name = "val" if (root / "val/images").is_dir() else (
            "valid" if (root / "valid/images").is_dir() else None
        )
        if train_ok and val_name:
            candidates.append((root, val_name))

    if not candidates:
        raise FileNotFoundError(
            "Không tìm thấy dataset có train/images, val/images hoặc valid/images, "
            "và test/images trong Kaggle Input."
        )
    return candidates[0]

DATA_ROOT, VAL_DIR_NAME = find_dataset_root()
DATA_YAML = WORK_ROOT / "dataset_6_model.yaml"

yaml_text = (
    f"path: '{DATA_ROOT}'\n"
    "train: train/images\n"
    f"val: {VAL_DIR_NAME}/images\n"
    "test: test/images\n"
    "nc: 6\n"
    "names: ['anten-4G', 'anten-5G', 'none', 'rrh', 'rru', 'viba']\n"
)
DATA_YAML.write_text(yaml_text, encoding="utf-8")

print("Dataset:", DATA_ROOT)
print("YAML:", DATA_YAML)
print(DATA_YAML.read_text())

Dataset: /kaggle/input/datasets/nonlam123/dataset-yolo/YOLO_Final_Dataset
YAML: /kaggle/working/dataset_6_model.yaml
path: '/kaggle/input/datasets/nonlam123/dataset-yolo/YOLO_Final_Dataset'
train: train/images
val: val/images
test: test/images
nc: 6
names: ['anten-4G', 'anten-5G', 'none', 'rrh', 'rru', 'viba']



In [12]:
# ===== [4] CHUẨN BỊ MÃ NGUỒN MAMBA-YOLO =====
# Ưu tiên repository đã được Add vào Kaggle Input.
# Nếu không có thì tự clone repository chính thức.

MAMBA_REPO = None

for train_script in INPUT_ROOT.rglob("mbyolo_train.py"):
    candidate = train_script.parent
    if (candidate / "ultralytics/nn/modules/mamba_yolo.py").is_file():
        MAMBA_REPO = candidate
        break

if MAMBA_REPO is None:
    candidate = WORK_ROOT / "Mamba-YOLO"
    if not candidate.exists():
        print("Không thấy repository trong Input -> đang clone Mamba-YOLO...")
        subprocess.run(
            [
                "git", "clone", "--depth", "1",
                "https://github.com/HZAI-ZJNU/Mamba-YOLO.git",
                str(candidate)
            ],
            check=True
        )
    MAMBA_REPO = candidate

assert (MAMBA_REPO / "ultralytics/nn/modules/mamba_yolo.py").is_file(), (
    f"Repository Mamba-YOLO không hợp lệ: {MAMBA_REPO}"
)

print("Mamba-YOLO repository:", MAMBA_REPO)

# Biên dịch/cài selective_scan cho đúng môi trường PyTorch/CUDA hiện tại.
selective_scan_dir = MAMBA_REPO / "selective_scan"
if not selective_scan_dir.is_dir():
    raise FileNotFoundError(f"Không thấy selective_scan tại {selective_scan_dir}")

print("Đang cài selective_scan; bước này có thể mất vài phút...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", str(selective_scan_dir)],
    check=True
)
print("Cài selective_scan hoàn tất.")

Mamba-YOLO repository: /kaggle/working/Mamba-YOLO
Đang cài selective_scan; bước này có thể mất vài phút...
Cài selective_scan hoàn tất.


In [13]:
# ===== [5] ĐO 5 MÔ HÌNH YOLO26 BẰNG ULTRALYTICS CHÍNH THỨC =====

from ultralytics.utils.torch_utils import get_flops

official_models = [(n, p) for n, p in MODELS if n != "Mamba-YOLO-B"]
results = []

# Warm-up GPU một lần.
warm_model = YOLO(official_models[0][1])
_ = warm_model.predict(
    np.zeros((640, 640, 3), dtype=np.uint8),
    imgsz=640,
    device=0,
    verbose=False
)
del warm_model
torch.cuda.empty_cache()

print(f"{'Model':14}{'Params(M)':>12}{'GFLOPs':>10}{'inf(ms)':>10}{'FPS':>9}")
print("-" * 55)

for name, pt in official_models:
    torch.cuda.empty_cache()
    model = YOLO(pt)

    params_m = sum(p.numel() for p in model.model.parameters()) / 1e6
    gflops = float(get_flops(model.model, imgsz=640))

    metrics = model.val(
        data=str(DATA_YAML),
        split="test",
        imgsz=640,
        device=0,
        workers=2,
        verbose=False
    )

    inf_ms = float(metrics.speed["inference"])
    fps = 1000.0 / inf_ms

    row = {
        "model": name,
        "params_M": round(params_m, 2),
        "gflops": round(gflops, 1),
        "inf_ms": round(inf_ms, 2),
        "fps": round(fps, 1)
    }
    results.append(row)

    print(
        f"{name:14}{params_m:>12.2f}{gflops:>10.1f}"
        f"{inf_ms:>10.2f}{fps:>9.1f}"
    )

YOLO_RESULT_JSON = WORK_ROOT / "yolo26_fps_results.json"
YOLO_RESULT_JSON.write_text(
    json.dumps(results, ensure_ascii=False, indent=2),
    encoding="utf-8"
)
print("\nĐã lưu:", YOLO_RESULT_JSON)

Model            Params(M)    GFLOPs   inf(ms)      FPS
-------------------------------------------------------
Ultralytics 8.4.108 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO26n summary (fused): 122 layers, 2,376,006 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 131.7±31.3 MB/s, size: 141.3 KB)
val: Scanning /kaggle/input/datasets/nonlam123/dataset-yolo/YOLO_Final_Dataset/test/labels... 206 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 206/206 847.3it/s 0.2s0.1s
val: /kaggle/input/datasets/nonlam123/dataset-yolo/YOLO_Final_Dataset/test/images/329_1744529411_TimePhoto_20250413_142955_jpg.rf.JMHQfqHtFTuDVAbKwhTF.jpg: 1 duplicate labels removed
val: /kaggle/input/datasets/nonlam123/dataset-yolo/YOLO_Final_Dataset/test/images/anh 1627_5_jpg.rf.4tP1cgO62oIeHPGyAqU4.jpg: 1 duplicate labels removed
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/nonlam123/dataset-yolo/YOLO_Final_Dataset/test is not writable, cache 

In [14]:
# ===== [6] ĐO MAMBA-YOLO-B TRONG TIẾN TRÌNH RIÊNG =====
# Tiến trình riêng ép Python dùng thư mục ultralytics của Mamba-YOLO,
# tránh xung đột với Ultralytics chính thức đang dùng cho YOLO26.

mamba_pt = dict(MODELS)["Mamba-YOLO-B"]
MAMBA_RESULT_JSON = WORK_ROOT / "mamba_fps_result.json"
MAMBA_SCRIPT = WORK_ROOT / "measure_mamba_subprocess.py"

mamba_script_text = '''
import os
import json
from pathlib import Path

import numpy as np
import torch

# PyTorch >= 2.6 mặc định torch.load(weights_only=True).
# Checkpoint Mamba-YOLO này lưu cả đối tượng DetectionModel, nên cần
# weights_only=False. Chỉ dùng với checkpoint do chính bạn tạo hoặc tin cậy.
_original_torch_load = torch.load

def _trusted_torch_load(*args, **kwargs):
    kwargs.setdefault("weights_only", False)
    return _original_torch_load(*args, **kwargs)

torch.load = _trusted_torch_load

from ultralytics import YOLO
from ultralytics.utils.torch_utils import get_flops

pt = os.environ["MAMBA_PT"]
data_yaml = os.environ["DATA_YAML"]
out_json = Path(os.environ["OUT_JSON"])

print("Ultralytics Mamba đang dùng:", __import__("ultralytics").__file__)
print("Checkpoint:", pt)

model = YOLO(pt)

_ = model.predict(
    np.zeros((640, 640, 3), dtype=np.uint8),
    imgsz=640,
    device=0,
    verbose=False
)
torch.cuda.synchronize()

params_m = sum(p.numel() for p in model.model.parameters()) / 1e6

try:
    gflops = float(get_flops(model.model, imgsz=640))
except Exception as exc:
    print("Cảnh báo: chưa tính được GFLOPs tự động:", repr(exc))
    gflops = None

metrics = model.val(
    data=data_yaml,
    split="test",
    imgsz=640,
    device=0,
    workers=2,
    verbose=False
)

inf_ms = float(metrics.speed["inference"])
fps = 1000.0 / inf_ms

row = {
    "model": "Mamba-YOLO-B",
    "params_M": round(params_m, 2),
    "gflops": None if gflops is None else round(gflops, 1),
    "inf_ms": round(inf_ms, 2),
    "fps": round(fps, 1)
}

out_json.write_text(
    json.dumps(row, ensure_ascii=False, indent=2),
    encoding="utf-8"
)
print("Kết quả Mamba:", row)
'''

MAMBA_SCRIPT.write_text(mamba_script_text, encoding="utf-8")

env = os.environ.copy()
env["MAMBA_PT"] = mamba_pt
env["DATA_YAML"] = str(DATA_YAML)
env["OUT_JSON"] = str(MAMBA_RESULT_JSON)

# PyTorch >= 2.6: buộc các callsite không truyền weights_only
# dùng chế độ legacy weights_only=False.
# Chỉ an toàn khi checkpoint là file do bạn tạo hoặc nguồn đáng tin cậy.
env["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
env.pop("TORCH_FORCE_WEIGHTS_ONLY_LOAD", None)

old_pythonpath = env.get("PYTHONPATH", "")
env["PYTHONPATH"] = str(MAMBA_REPO) + (
    os.pathsep + old_pythonpath if old_pythonpath else ""
)

subprocess.run(
    [sys.executable, str(MAMBA_SCRIPT)],
    cwd=str(MAMBA_REPO),
    env=env,
    check=True
)

print("Đã lưu:", MAMBA_RESULT_JSON)

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/kaggle/working/Mamba-YOLO/ultralytics/nn/modules/common_utils_mbyolo.py:103: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd
/kaggle/working/Mamba-YOLO/ultralytics/nn/modules/common_utils_mbyolo.py:130: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_bwd


Ultralytics Mamba đang dùng: /kaggle/working/Mamba-YOLO/ultralytics/__init__.py
Checkpoint: /kaggle/input/datasets/nonlam123/best-6-model/BEST_6_MODEL/best_bestmamba.pt
Ultralytics YOLOv8.2.29 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Mamba-YOLO-B summary: 388 layers, 21797058 parameters, 0 gradients, 49.6 GFLOPs


val: Scanning /kaggle/input/datasets/nonlam123/dataset-yolo/YOLO_Final_Dataset/test/labels... 206 images, 0 backgrounds, 0 corrupt: 100%|██████████| 206/206 [00:00<00:00, 849.84it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95):   0%|          | 0/13 [00:00<?, ?it/s]

val: WARNING ⚠️ /kaggle/input/datasets/nonlam123/dataset-yolo/YOLO_Final_Dataset/test/images/329_1744529411_TimePhoto_20250413_142955_jpg.rf.JMHQfqHtFTuDVAbKwhTF.jpg: 1 duplicate labels removed
val: WARNING ⚠️ /kaggle/input/datasets/nonlam123/dataset-yolo/YOLO_Final_Dataset/test/images/anh 1627_5_jpg.rf.4tP1cgO62oIeHPGyAqU4.jpg: 1 duplicate labels removed
val: WARNING ⚠️ Cache directory /kaggle/input/datasets/nonlam123/dataset-yolo/YOLO_Final_Dataset/test is not writeable, cache not saved.


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 13/13 [00:12<00:00,  1.03it/s]


                   all        206       1675      0.842       0.73      0.802      0.534
Speed: 1.2ms preprocess, 53.0ms inference, 0.0ms loss, 1.3ms postprocess per image
Results saved to /kaggle/working/Mamba-YOLO/runs/detect/val
Kết quả Mamba: {'model': 'Mamba-YOLO-B', 'params_M': 21.8, 'gflops': 49.6, 'inf_ms': 53.0, 'fps': 18.9}
Đã lưu: /kaggle/working/mamba_fps_result.json


In [15]:
# ===== [7] GỘP VÀ XUẤT BẢNG 6 MÔ HÌNH =====

official_rows = json.loads(YOLO_RESULT_JSON.read_text(encoding="utf-8"))
mamba_row = json.loads(MAMBA_RESULT_JSON.read_text(encoding="utf-8"))

all_rows = official_rows + [mamba_row]
order_index = {name: i for i, name in enumerate(MODEL_ORDER)}
all_rows.sort(key=lambda r: order_index[r["model"]])

df = pd.DataFrame(all_rows)
df = df[["model", "params_M", "gflops", "inf_ms", "fps"]]

CSV_PATH = WORK_ROOT / "fps_6_models_T4.csv"
JSON_PATH = WORK_ROOT / "fps_6_models_T4.json"

df.to_csv(CSV_PATH, index=False, encoding="utf-8-sig")
JSON_PATH.write_text(
    json.dumps(all_rows, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

display(df)

print("\n=== DÒNG JSON ĐỂ SAO CHÉP ===")
print(json.dumps(
    [(r["model"], r["inf_ms"], r["fps"]) for r in all_rows],
    ensure_ascii=False
))

print("\nĐã xuất:")
print(" ", CSV_PATH)
print(" ", JSON_PATH)

,model,params_M,gflops,inf_ms,fps
0,YOLO26n,2.51,5.8,3.34,299.0
1,YOLO26s,9.95,22.5,7.87,127.1
2,YOLO26m,21.78,74.8,19.42,51.5
3,YOLO26l,26.19,93.2,23.84,41.9
4,YOLO26x,58.82,208.6,49.15,20.3
5,Mamba-YOLO-B,21.80,49.6,53.00,18.9



=== DÒNG JSON ĐỂ SAO CHÉP ===
[["YOLO26n", 3.34, 299.0], ["YOLO26s", 7.87, 127.1], ["YOLO26m", 19.42, 51.5], ["YOLO26l", 23.84, 41.9], ["YOLO26x", 49.15, 20.3], ["Mamba-YOLO-B", 53.0, 18.9]]

Đã xuất:
  /kaggle/working/fps_6_models_T4.csv
  /kaggle/working/fps_6_models_T4.json
